# Unit 7 — Corporate Actions & Survivorship Bias 🔴 CRITICAL
**Phase 2 | Project Prometheus**

---

## 🎯 Why This Matters in Quant Finance

This is where **amateur quants fail silently**.

A bad backtest shows 50% annual returns. The real strategy loses money.

Two mechanisms cause this:
1. **Incorrect corporate action adjustments** — price series look like they crashed when they didn't (or surged when they didn't)
2. **Survivorship bias** — you only test on companies that survived, inflating apparent returns by 5–10% per year

> 🎯 Interview certainty: *"How do you handle corporate actions?"* — you need a confident, specific answer.


---
## 1️⃣ Stock Splits

### What happens in a 2-for-1 split?
- Share count doubles, price halves
- **Total market cap is unchanged**
- Historical prices **before** the split must be adjusted DOWN

### Why adjust historical prices?
Without adjustment, your price series looks like this:

```
Aug 28 2020:  $499   ← pre-split (unadjusted)
Aug 31 2020:  $124   ← AAPL 4-for-1 split date
```

That -75% drop is **fake** — no shareholder lost money. Your strategy would short AAPL on that signal.

### Adjustment formula
```
Adjusted price (pre-split) = Unadjusted price / Split ratio
```

For a 4-for-1 split: divide all pre-split prices by 4.

### Real examples to know
| Ticker | Date | Split |
|--------|------|-------|
| AAPL | 31 Aug 2020 | 4-for-1 |
| TSLA | 31 Aug 2020 | 5-for-1 |
| NVDA | 10 Jun 2024 | 10-for-1 |
| AMZN | 3 Jun 2022 | 20-for-1 |


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# AAPL: compare adjusted vs unadjusted around the Aug 2020 split
aapl_adj = yf.download('AAPL', start='2020-08-01', end='2020-09-15',
                        auto_adjust=True)['Close'].squeeze()
aapl_raw = yf.download('AAPL', start='2020-08-01', end='2020-09-15',
                        auto_adjust=False)['Close'].squeeze()

comparison = pd.DataFrame({'Adjusted': aapl_adj, 'Unadjusted': aapl_raw})
comparison['Ratio'] = comparison['Unadjusted'] / comparison['Adjusted']

print("AAPL around 4-for-1 split (31 Aug 2020):")
print(comparison.round(2))
print("\n→ Ratio should be ~4.0 before split, ~1.0 after")


In [ ]:
# Pull split and dividend history directly from yfinance
aapl = yf.Ticker('AAPL')
tsla = yf.Ticker('TSLA')

print("AAPL Splits:")
print(aapl.splits)

print("\nTSLA Splits:")
print(tsla.splits)

print("\nAAPL Dividends (last 5):")
print(aapl.dividends.tail(5))


---
## 2️⃣ Dividends — Price Return vs Total Return

| Measure | Includes Dividends? | Use When |
|---------|-------------------|----------|
| Price Return | ❌ No | Comparing capital gains only |
| Total Return | ✅ Yes (reinvested) | Evaluating full investor experience |

**Total return is always higher** for dividend-paying stocks. Comparing price returns across high-yield vs growth stocks is apples-to-oranges.

### Dividend adjustment
On ex-dividend date, the stock price drops by approximately the dividend amount.
To make the historical series continuous, adjust all pre-dividend prices down:

```
Adjustment factor = (Price_before - Dividend) / Price_before
```

`auto_adjust=True` in yfinance handles both splits AND dividends for you.


In [ ]:
# Demonstrate total return vs price return for a dividend payer
# JPM is a consistent dividend payer — good example

jpm_adj = yf.download('JPM', start='2018-01-01', auto_adjust=True)['Close'].squeeze()
jpm_raw = yf.download('JPM', start='2018-01-01', auto_adjust=False)['Close'].squeeze()

# Calculate cumulative returns
price_return = (jpm_raw.iloc[-1] / jpm_raw.iloc[0] - 1) * 100
total_return = (jpm_adj.iloc[-1] / jpm_adj.iloc[0] - 1) * 100

print(f"JPM Price Return (unadjusted): {price_return:.1f}%")
print(f"JPM Total Return (adjusted):   {total_return:.1f}%")
print(f"Dividend contribution: {total_return - price_return:.1f}%")
print("\n→ The gap IS the value of dividends reinvested")


---
## 3️⃣ Survivorship Bias

### What is it?
If you build a backtest using **today's S&P 500 constituents**, you're only testing companies that:
- Survived to today
- Grew large enough to be in the index

You're **excluding** companies that:
- Went bankrupt (Lehman Brothers, Enron)
- Were delisted or acquired
- Shrank and dropped out of the index

### How much does it matter?
Studies estimate survivorship bias inflates apparent returns by **1.5% to 5%+ per year** depending on the universe. For small caps, it can be even larger.

### Why it's insidious
Your strategy looks profitable in backtest. You deploy it. It underperforms. The backtest was testing on "winners only" — you could have picked any stock in the universe and shown positive returns.

### How to handle it (awareness level for Phase 2)
| Approach | Description |
|----------|-------------|
| Point-in-time index data | Use index constituents as they were at each historical date |
| CRSP database | Academic standard — includes delisted stocks |
| Explicit delisted tickers | Add known failures to your test universe |
| Acknowledge the limitation | At minimum, document it in your backtest assumptions |


In [ ]:
# Simple survivorship bias demonstration
# Strategy: invest equally in current S&P 500 tech giants

survivors = ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'META']

# These were all massively successful — obvious hindsight selection
prices = yf.download(survivors, start='2010-01-01', auto_adjust=True)['Close']
returns = prices.pct_change().dropna()

# Equal weight portfolio
portfolio = returns.mean(axis=1)
cumulative = (1 + portfolio).cumprod()

total_ret = (cumulative.iloc[-1] - 1) * 100
years = (cumulative.index[-1] - cumulative.index[0]).days / 365.25
cagr = ((cumulative.iloc[-1]) ** (1/years) - 1) * 100

print(f"'Strategy': Buy top 5 tech giants in 2010 (selected in 2024)")
print(f"Total Return: {total_ret:.0f}%")
print(f"CAGR: {cagr:.1f}%")
print("\n→ This is pure hindsight. In 2010 you didn't know these would win.")
print("→ A real backtest would include Nokia, BlackBerry, Yahoo, Kodak...")


In [ ]:
# Check if a ticker has been delisted (returns empty or short history)
def check_ticker_status(ticker, start='2010-01-01'):
    try:
        data = yf.download(ticker, start=start, auto_adjust=True, progress=False)['Close']
        if len(data) == 0:
            return "No data — likely delisted"
        last_date = data.index[-1]
        days_since_last = (pd.Timestamp.today() - last_date).days
        if days_since_last > 30:
            return f"Last data: {last_date.date()} — possibly delisted"
        return f"Active — {len(data)} trading days of data"
    except Exception as e:
        return f"Error: {e}"

# Test some known cases
test_tickers = ['AAPL', 'LEHMQ', 'SPCE']  # Apple, Lehman (bankrupt), Virgin Galactic
for t in test_tickers:
    print(f"{t}: {check_ticker_status(t)}")


---
## ✅ Self-Check Questions

1. What's the difference between split-adjusted and dividend-adjusted prices?
   > *Split-adjusted: price scaled for share count changes. Dividend-adjusted: price also scaled for cash distributions. `auto_adjust=True` does both.*

2. How do you detect if a stock had a 4-for-1 split on a specific date?
   > *Check `yf.Ticker(tick).splits` — returns a Series of split ratios by date. Or compare adjusted vs unadjusted price ratio (should be ~4.0 before the date).*

3. Why does survivorship bias matter for backtesting?
   > *If you only test on today's index members, you exclude all failures. Every surviving stock looks like a winner in hindsight. Returns are artificially inflated.*

4. What's the formula for adjusting historical prices after a 2-for-1 split?
   > *Adjusted price = Unadjusted price / 2 (for all prices before the split date)*

5. How do you handle total return vs price return?
   > *Use `auto_adjust=True` in yfinance for total return. Use `auto_adjust=False` for price return only.*

---
## 🧪 Optional Tasks

- Download AAPL adjusted vs unadjusted for 2020. Plot both. The split is clearly visible in unadjusted data.
- Compare total return vs price return for a high-dividend stock (e.g., T, VZ, or KO) over 10 years.
- Download TSLA splits. Verify the 5-for-1 split on 31 Aug 2020.
- Build the `check_ticker_status()` function above. Test it on 10 tickers including some you suspect might be delisted.
